# OBJECTIVE
The objective of this methodology  is to reconcile bilateral trade data reported in UN Comtrade to establish a single, consistent trade value between partners. By addressing discrepancies and measuring the reliability of reported trade flows, this approach enhances the accuracy of trade records, resulting in a cohesive dataset for international economic analysis.

# Installation of complexity libraries

In [1]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip -q install econci

In [3]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import econci

## Load Trade data

In [4]:
root_folder = '/content/drive/MyDrive/Auto Research Team Folder/Trade reconciliation/Trade  data/UN comtrade/2021'
trade_name = "HS_6_Digit_2021.csv"
df_2021 = pd.read_csv(f'{root_folder}/{trade_name}',dtype = {'cmdCode':str,'H0':str})

In [5]:
# drop world as a partner
df_2021 = df_2021[df_2021.partnerDesc != 'World']

In [6]:
# split dataframes based on flowcode
imports_df = df_2021[df_2021['flowCode'] == 'M']
exports_df = df_2021[df_2021['flowCode'] == 'X']

In [7]:
imports_df = imports_df[['period','reporterISO','reporterDesc','partnerISO', 'partnerDesc','cmdCode', 'H0','cifvalue']]
imports_df = imports_df.rename(columns={'reporterDesc': 'importer','reporterISO':'importerISO', 'partnerDesc': 'exporter','partnerISO':'exporterISO','cifvalue':'import_value'})

In [8]:
exports_df = exports_df[['period','reporterISO','reporterDesc', 'partnerISO', 'partnerDesc','cmdCode','H0','fobvalue']]
exports_df = exports_df.rename(columns={'reporterDesc': 'exporter','reporterISO':'exporterISO', 'partnerDesc': 'importer','partnerISO':'importerISO', 'fobvalue': 'export_value'})

In [9]:
imports_df.sample(n=5)

,period,importerISO,importer,exporterISO,exporter,cmdCode,H0,import_value
11463485,2021,CHN,China,ESP,Spain,761610,761610,685063.000
12384822,2021,BGR,Bulgaria,ITA,Italy,841319,841319,177039.196
16338614,2021,NER,Niger,USA,USA,870422,870422,111862.895
4694350,2021,BLR,Belarus,ARM,Armenia,391890,391890,100.000
12956878,2021,FIN,Finland,CHE,Switzerland,842449,842481,5227.712


In [10]:
cifvalue_sum = imports_df['import_value'].sum()
print(f"The sum of cifvalue is: {cifvalue_sum}")

The sum of cifvalue is: 20882207860012.316


In [11]:
fobvalue_sum = exports_df['export_value'].sum()
print(f"The sum of fobvalue is: {fobvalue_sum}")

The sum of fobvalue is: 21687056340877.38


In [12]:
print(exports_df['importer'].nunique())
print(imports_df['exporter'].nunique())
print(imports_df['importer'].nunique())
print(exports_df['exporter'].nunique())

245
244
167
167


process dataframe

In [13]:
imports_df.reset_index(drop=True, inplace=True)
exports_df.reset_index(drop=True, inplace=True)

In [14]:
exports_df.head()

,period,exporterISO,exporter,importerISO,importer,cmdCode,H0,export_value
0,2021,ATG,Antigua and Barbuda,DMA,Dominica,010110,010111,555.556
1,2021,NOR,Norway,SWE,Sweden,010121,010111,303396.610
2,2021,NOR,Norway,NLD,Netherlands,010121,010111,2379.511
3,2021,NOR,Norway,IRL,Ireland,010121,010111,473797.999
4,2021,NOR,Norway,FRA,France,010121,010111,5983.702


In [15]:
imports_df.head()

,period,importerISO,importer,exporterISO,exporter,cmdCode,H0,import_value
0,2021,ZAF,South Africa,NAM,Namibia,010121,010111,0.0
1,2021,ZAF,South Africa,S19,"Other Asia, nes",010121,010111,0.0
2,2021,ZAF,South Africa,LSO,Lesotho,010121,010111,0.0
3,2021,ZAF,South Africa,IRL,Ireland,010121,010111,0.0
4,2021,ZAF,South Africa,IDN,Indonesia,010121,010111,0.0


In [16]:
imports_df.isna().sum()

,0
period,0
importerISO,0
importer,0
exporterISO,0
exporter,0
cmdCode,0
H0,0
import_value,201273


In [17]:
exports_df.isna().sum()

,0
period,0
exporterISO,0
exporter,0
importerISO,0
importer,0
cmdCode,0
H0,0
export_value,1


# merge imports and exports data

In [18]:
trade_raw = pd.merge(imports_df, exports_df, on=['period','exporter','exporterISO','cmdCode', 'importer','importerISO',  'H0'], how='outer')
print(trade_raw.shape)
trade_raw.head()

(12031905, 9)


,period,importerISO,importer,exporterISO,exporter,cmdCode,H0,import_value,export_value
0,2021,IRL,Ireland,AFG,Afghanistan,010121,010111,17.741,NaN
1,2021,ESP,Spain,AFG,Afghanistan,010121,010111,4491.374,NaN
2,2021,TJK,Tajikistan,AFG,Afghanistan,010129,010119,1100.000,NaN
3,2021,UZB,Uzbekistan,AFG,Afghanistan,010129,010119,9700.000,NaN
4,2021,TJK,Tajikistan,AFG,Afghanistan,010229,010290,5800.000,NaN


In [19]:
trade_raw['exporter'] = trade_raw['exporter'].replace('Sint Maarten', 'Saint Maarten')
trade_raw['importer'] = trade_raw['importer'].replace('Sint Maarten', 'Saint Maarten')

In [20]:
print(trade_raw['importerISO'].nunique())
trade_raw['exporterISO'].nunique()

244


244

In [21]:
trade_raw.isna().sum()

,0
period,0
importerISO,0
importer,0
exporterISO,0
exporter,0
cmdCode,0
H0,0
import_value,2910561
export_value,3943784


In [22]:
print(trade_raw['cmdCode'].nunique())
print(trade_raw['H0'].nunique())

5599
4645


In [23]:
# Fill NaN values in 'H0' column with corresponding values from 'cmdCode' column
#trade_raw['H0'] = trade_raw['H0'].fillna(trade_raw['cmdCode'])

In [24]:
trade_raw['import_value'].fillna(0, inplace=True)
trade_raw['export_value'].fillna(0, inplace=True)

<ipython-input-24-462dd08f7516>:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  trade_raw['import_value'].fillna(0, inplace=True)
<ipython-input-24-462dd08f7516>:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usi

In [25]:
trade_raw.isna().sum()

,0
period,0
importerISO,0
importer,0
exporterISO,0
exporter,0
cmdCode,0
H0,0
import_value,0
export_value,0


## list of 226 countries included

In [26]:
exporter_ISO = ['ABW', 'AFG', 'AGO', 'AIA', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', 'ASM', 'ATF', 'ATG', 'S19', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BES', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS',  'BIH', 'BLM', 'BLR', 'BLZ', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF',  'CAN', 'CCK', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COG', 'COK', 'COL', 'COM',  'CPV', 'CRI', 'CUB', 'CUW', 'CXR', 'CYM', 'CYP', 'CZE', 'DEU', 'DJI', 'DMA', 'DNK',  'DOM', 'DZA', 'ECU', 'EGY', 'ERI', 'ESP', 'EST', 'ETH', 'FIN', 'FJI', 'FLK', 'FRA',  'FSM', 'GAB', 'GBR', 'GEO', 'GHA', 'GIB', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GRD',  'GRL', 'GTM', 'GUM', 'GUY', 'HKG', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IND', 'IOT',  'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ',  'KHM', 'KIR', 'KNA', 'KOR', 'KWT', 'LAO', 'LBN', 'LBR', 'LBY', 'LCA', 'LKA', 'LSO',  'LTU', 'LUX', 'LVA', 'MAC', 'MAR', 'MDA', 'MDG', 'MDV', 'MEX', 'MHL', 'MKD', 'MLI',  'MLT', 'MMR', 'MNE', 'MNG', 'MNP', 'MOZ', 'MRT', 'MSR', 'MUS', 'MWI', 'MYS', 'NAM',  'NCL', 'NER', 'NFK', 'NGA', 'NIC', 'NIU', 'NLD', 'NOR', 'NPL', 'NRU', 'NZL', 'OMN',  'PAK', 'PAN', 'PCN', 'PER', 'PHL', 'PLW', 'PNG', 'POL', 'PRK', 'PRT', 'PRY', 'PSE',  'PYF', 'QAT', 'ROU', 'RUS', 'RWA', 'SAU', 'SDN', 'SEN', 'SGP', 'SHN', 'SLB',  'SLE', 'SLV', 'SMR', 'SOM', 'SPM', 'SRB', 'SSD', 'STP', 'SUR', 'SVK', 'SVN', 'SWE',  'SWZ', 'SXM', 'SYC', 'SYR', 'TCA', 'TCD', 'TGO', 'THA', 'TJK', 'TKL', 'TKM', 'TLS',  'TON', 'TTO', 'TUN', 'TUR', 'TUV', 'TZA', 'UGA', 'UKR', 'URY', 'USA', 'UZB', 'VCT',  'VEN', 'VGB', 'VNM', 'VUT', 'WLF', 'WSM', 'YEM', 'ZAF', 'ZMB', 'ZWE']
exporter_ISO = pd.DataFrame(exporter_ISO, columns=['exporterISO'])

In [27]:
importer_ISO = ['ABW', 'AFG', 'AGO', 'AIA', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', 'ASM', 'ATF', 'ATG', 'S19', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BES', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS',  'BIH', 'BLM', 'BLR', 'BLZ', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF',  'CAN', 'CCK', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COG', 'COK', 'COL', 'COM',  'CPV', 'CRI', 'CUB', 'CUW', 'CXR', 'CYM', 'CYP', 'CZE', 'DEU', 'DJI', 'DMA', 'DNK',  'DOM', 'DZA', 'ECU', 'EGY', 'ERI', 'ESP', 'EST', 'ETH', 'FIN', 'FJI', 'FLK', 'FRA',  'FSM', 'GAB', 'GBR', 'GEO', 'GHA', 'GIB', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GRD',  'GRL', 'GTM', 'GUM', 'GUY', 'HKG', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IND', 'IOT',  'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ',  'KHM', 'KIR', 'KNA', 'KOR', 'KWT', 'LAO', 'LBN', 'LBR', 'LBY', 'LCA', 'LKA', 'LSO',  'LTU', 'LUX', 'LVA', 'MAC', 'MAR', 'MDA', 'MDG', 'MDV', 'MEX', 'MHL', 'MKD', 'MLI',  'MLT', 'MMR', 'MNE', 'MNG', 'MNP', 'MOZ', 'MRT', 'MSR', 'MUS', 'MWI', 'MYS', 'NAM',  'NCL', 'NER', 'NFK', 'NGA', 'NIC', 'NIU', 'NLD', 'NOR', 'NPL', 'NRU', 'NZL', 'OMN',  'PAK', 'PAN', 'PCN', 'PER', 'PHL', 'PLW', 'PNG', 'POL', 'PRK', 'PRT', 'PRY', 'PSE',  'PYF', 'QAT', 'ROU', 'RUS', 'RWA', 'SAU', 'SDN', 'SEN', 'SGP', 'SHN', 'SLB',  'SLE', 'SLV', 'SMR', 'SOM', 'SPM', 'SRB', 'SSD', 'STP', 'SUR', 'SVK', 'SVN', 'SWE',  'SWZ', 'SXM', 'SYC', 'SYR', 'TCA', 'TCD', 'TGO', 'THA', 'TJK', 'TKL', 'TKM', 'TLS',  'TON', 'TTO', 'TUN', 'TUR', 'TUV', 'TZA', 'UGA', 'UKR', 'URY', 'USA', 'UZB', 'VCT',  'VEN', 'VGB', 'VNM', 'VUT', 'WLF', 'WSM', 'YEM', 'ZAF', 'ZMB', 'ZWE']
importer_ISO = pd.DataFrame(importer_ISO, columns=['importerISO'])

In [28]:
print(len(exporter_ISO))
print(len(importer_ISO))

226
226


In [29]:
trade_pre = pd.merge(trade_raw, exporter_ISO, on=['exporterISO'], how='right')
print(trade_pre.shape)
trade_pre.head()

(11940899, 9)


,period,importerISO,importer,exporterISO,exporter,cmdCode,H0,import_value,export_value
0,2021,COL,Colombia,ABW,Aruba,010121,010111,4139.0,12000.000
1,2021,CUW,Curaçao,ABW,Aruba,010129,010119,0.0,2793.296
2,2021,NLD,Netherlands,ABW,Aruba,010190,010120,0.0,1955.307
3,2021,CUW,Curaçao,ABW,Aruba,010410,010410,0.0,300.000
4,2021,COL,Colombia,ABW,Aruba,010511,010511,0.0,6432.961


In [30]:
trade_data = pd.merge(trade_pre, importer_ISO, on=['importerISO'], how='right')
print(trade_data.shape)
trade_data.head()

(11874915, 9)


,period,importerISO,importer,exporterISO,exporter,cmdCode,H0,import_value,export_value
0,2021,ABW,Aruba,ARE,United Arab Emirates,090230,090230,0.0,164019.464
1,2021,ABW,Aruba,ARE,United Arab Emirates,090240,090240,0.0,78624.096
2,2021,ABW,Aruba,ARE,United Arab Emirates,210690,210690,0.0,6301.429
3,2021,ABW,Aruba,ARE,United Arab Emirates,240220,240220,0.0,7019196.536
4,2021,ABW,Aruba,ARE,United Arab Emirates,240311,240310,0.0,1694.407


In [31]:
# Round off import_value and export_value to the next whole number
trade_data['import_value'] = trade_data['import_value'].round()
trade_data['export_value'] = trade_data['export_value'].round()

In [32]:
trade_data.drop(['exporter', 'importer'], axis=1, inplace=True)

In [33]:
print(trade_data['exporterISO'].nunique())
print(trade_data['importerISO'].nunique())
print(trade_data['H0'].nunique())
print(trade_data['cmdCode'].nunique())

226
226
4645
5599


# what is the raw total values?

In [34]:
total_export_value = trade_data['export_value'].sum()
total_import_value = trade_data['import_value'].sum()

print(f"Total Export Value: {total_export_value}")
print(f"Total Import Value: {total_import_value}")

Total Export Value: 20926647950446.0
Total Import Value: 20593901264253.0


In [35]:
trade_df = trade_data.copy()

In [36]:
trade_df.head()

,period,importerISO,exporterISO,cmdCode,H0,import_value,export_value
0,2021,ABW,ARE,090230,090230,0.0,164019.0
1,2021,ABW,ARE,090240,090240,0.0,78624.0
2,2021,ABW,ARE,210690,210690,0.0,6301.0
3,2021,ABW,ARE,240220,240220,0.0,7019197.0
4,2021,ABW,ARE,240311,240310,0.0,1694.0


In [37]:
trade_df.sort_values(by='H0', ascending=True, inplace=True)

In [38]:
trade_df.head()

,period,importerISO,exporterISO,cmdCode,H0,import_value,export_value
10118696,2021,SUR,POL,999999,-00001,14707.0,0.0
3172149,2021,DJI,JPN,999999,-00001,20050.0,0.0
8596131,2021,PAK,DJI,999999,-00001,0.0,87.0
8092993,2021,NLD,DJI,999999,-00001,0.0,167924.0
3172505,2021,DJI,KEN,999999,-00001,50241.0,0.0


In [39]:
# drop unclassified items
trade_df = trade_df[trade_df['cmdCode'] != '999999']

In [41]:
trade_df.drop(columns=['cmdCode'], inplace=True)# drop unnecessary columns

In [42]:
trade_df.head()

,period,importerISO,exporterISO,H0,import_value,export_value
4133947,2021,FRA,MUS,010111,13720.0,0.0
5855391,2021,ITA,ISL,010111,0.0,3963.0
11406891,2021,USA,PAN,010111,102153.0,12000.0
10308369,2021,SVN,NLD,010111,27402.0,27506.0
5512087,2021,IRL,SVK,010111,6491.0,0.0


In [38]:
#trade_df['H0'] = trade_df['H0'].astype(str).str.zfill(6)  # Pad with leading zeros

In [43]:
# Convert H0 (6-digit HS code) to HS4 (4-digit HS code)
trade_df['HS4_code'] = trade_df['H0'].str[:4]

In [44]:
# drop rows with 0 commodity code 9999
trade_df = trade_df[trade_df['HS4_code'] != '9999']

In [45]:
trade_df.sort_values(by='HS4_code', ascending=True).head()

,period,importerISO,exporterISO,H0,import_value,export_value,HS4_code
4133947,2021,FRA,MUS,010111,13720.0,0.0,0101
1745440,2021,CAN,AUS,010119,0.0,0.0,0101
1914457,2021,CHE,AUT,010119,478405.0,91101.0,0101
1026302,2021,BEL,URY,010119,0.0,9000.0,0101
9405458,2021,ROU,SVK,010119,15365.0,0.0,0101


In [46]:
# drop rows where exporterISO is equal to importerISO
trade_df = trade_df[trade_df['exporterISO'] != trade_df['importerISO']]

In [47]:
print(trade_df['importerISO'].nunique())
print(trade_df['exporterISO'].nunique())
print(trade_df['H0'].nunique())
print(trade_df['HS4_code'].nunique())

226
226
4643
1217


In [48]:
print(trade_df.isna().sum())

period          0
importerISO     0
exporterISO     0
H0              0
import_value    0
export_value    0
HS4_code        0
dtype: int64


In [49]:
trade_df.sample(n=5)

,period,importerISO,exporterISO,H0,import_value,export_value,HS4_code
4659858,2021,GRC,FIN,902680,27273.0,204978.0,9026
6064059,2021,JPN,HUN,392321,0.0,1.0,3923
460979,2021,S19,CHN,846691,11826045.0,9822473.0,8466
7174230,2021,MDG,ARE,300610,124581.0,0.0,3006
7092379,2021,MAR,NLD,440110,723.0,0.0,4401


In [49]:
total_export_value = trade_df['export_value'].sum()
total_import_value = trade_df['import_value'].sum()

print(f"Total Export Value: {total_export_value}")
print(f"Total Import Value: {total_import_value}")

Total Export Value: 20319800190403.0
Total Import Value: 20038110985277.0


In [50]:
root_folder = "/content/drive/MyDrive/Auto Research Team Folder/Trade reconciliation/Trade  data/UN comtrade/2021"
file_name= "trade_corr_df.csv"
trade_df.to_csv(f'{root_folder}/{file_name}', index=False)

## Trade reconciliation
steps:


1.   Calclulate accuracy level
2.   Calculate of total trade at commodity level for both exports and imports
3.   Calculate Accurately matched exports and imports
4.   Calculate Reliability indexes based only on the accurate transcations
5.   Reconcile Trade values based on reliability indexes



In [ ]:
# Replace zero values with (1e-10) easy calculations.
for col in ['export_value', 'import_value']:
    trade_df[col] = np.where(trade_df[col] == 0, 1e-10, trade_df[col])

In [ ]:
trade_df.sample(n=5)

,period,importerISO,exporterISO,H0,import_value,export_value,HS4_code
10873780,2022,TTO,GTM,761690,1.000000e-10,1.171000e+03,7616
8354623,2022,NLD,TTO,890190,2.632622e+06,1.000000e-10,8901
3095285,2022,DEU,NLD,180632,1.280666e+07,1.137549e+07,1806
1286170,2022,BHR,IDN,610332,1.121000e+03,1.000000e-10,6103
8080496,2022,NGA,USA,846420,7.125800e+04,8.850500e+04,8464


###  step 1: calculate Accuracy level



In [ ]:
def calculate_accuracy_level(import_value, export_value):
    accuracy_level = abs(import_value - export_value) / import_value * 100
    return accuracy_level

In [ ]:
trade_df.loc[:, 'accuracy_level'] = trade_df.apply(lambda row: calculate_accuracy_level(row['import_value'], row['export_value']), axis=1)

### Step 2: Calculation of total trade at commodity level
calculate the total trade reported by the importer i for commodity s and likewise for exporter.

In [ ]:
trade_df['total_imports'] = trade_df.groupby(['importerISO','H0'])['import_value'].transform('sum')
trade_df['total_exports'] = trade_df.groupby(['exporterISO','H0'])['export_value'].transform('sum')
trade_df.head()

In [ ]:
# Reset zero value
for col in ['export_value', 'import_value']:
    trade_df[col] = np.where(trade_df[col] ==1e-10, 0, trade_df[col])

###  Step 3: calculately accurately matched exports and imports
A threshold level must be established. It is the difference as a percentage between reported exports and reported imports. This level as been established at 20%. if it less than 20% its considered a match.



In [ ]:
# Set the threshold
threshold = 20

Add a new column, 'accurate_import', to trade_df, which flags imports as "accurate" if the discrepancy (measured by accuracy_level) is within a specified threshold.

In [ ]:
trade_df['accurate_import'] = trade_df.apply(lambda row: row['import_value'] if row['accuracy_level'] <= threshold else 0, axis=1)

Add a new column, 'accurate_export', to trade_df, which flags exports as "accurate" if the discrepancy (measured by accuracy_level) is within a specified threshold.


In [ ]:
trade_df['accurate_export'] = trade_df.apply(lambda row: row['export_value'] if row['accuracy_level'] <= threshold else 0, axis=1)

In [ ]:
trade_df.sample(n=5)

### step 4: calculate reliability indexes based on share of accurate transactions

**IMPORTER RELIABILITY INDEX**

The importer-commodity reliability index is calculated by first grouping the data in the DataFrame based on each importer and commodity code (cmdCode). For each group, a lambda function is applied to calculate the  percentage of accurate imports (total_accurate_imports) relative to total imports (total_imports)

1.  summming  of accurate_import values in the group

2. Dividing sum of accurate_import_value by total number of exports in that group.

3. Multiplying the result by 100 to express the reliability as a percentage.


In [ ]:
trade_df['total_accurate_imports'] = trade_df.groupby(['importerISO', 'H0'])['accurate_import'].transform('sum')
trade_df['total_accurate_exports'] = trade_df.groupby(['exporterISO', 'H0'])['accurate_export'].transform('sum')

In [ ]:
trade_df['importer_commodity_reliability'] = (
    trade_df.groupby(['importerISO', 'H0'])['total_accurate_imports'].transform(lambda x: (x / trade_df.loc[x.index, 'total_imports']) * 100))


**EXPORTER RELIABILITY INDEX**

The exporter-commodity reliability index is calculated by first grouping the data in the DataFrame based on each exporter and commodity code (cmdCode). For each group, a lambda function is applied to calculate the  percentage of accurate exports (total_accurate_exports) relative to total exports (total_exports)

1. Summing the accurate_export values in the group

2. Dividing the sum of accurate exports by the total exports in that group.

3. Multiplying the result by 100 to express the reliability as a percentage.

In [ ]:
trade_df['exporter_commodity_reliability'] = (
    trade_df.groupby(['exporterISO', 'H0'])['total_accurate_exports'].transform(lambda x: (x / trade_df.loc[x.index, 'total_exports']) * 100))

make a copy just incase you want to retrace a step

In [ ]:
trade_df_re = trade_df.copy()

In [ ]:
trade_df = trade_df[['period','HS4_code','H0', 'exporterISO', 'importerISO','export_value', 'import_value','importer_commodity_reliability', 'exporter_commodity_reliability']]

### Step 6: Based on the reliability indexes calculated get the most accurate transcation.
 **Example**

 If the reliability index for the importer is 90% and for the exporter is 75%, the function would return the importer_value since the importer's data is considered more reliable.

 By choosing the value from the partner with the higher reliability index, the function tries to achieve a more accurate representation of trade values.

In [ ]:
def reconcile_trade(import_value, export_value, importer_commodity_reliability, exporter_commodity_reliability):
    """
    Reconciles trade data by considering the exporter with recorded data,
    otherwise considering the more reliable partner's data.
    """
    if import_value == 0 and export_value == 0:
        return 0
    # Check if import value is zero and export value is non-zero
    if import_value == 0 and export_value != 0:
        return export_value

    # If both values are non-zero, consider reliability
    if importer_commodity_reliability >= exporter_commodity_reliability:
        return import_value  # Trust importer's data more
    else:
        return export_value  # Trust exporter's data more

In [ ]:
trade_df.loc[:, 'reconciled_value'] = trade_df.apply(
    lambda x: reconcile_trade( x['import_value'],x['export_value'],x['importer_commodity_reliability'],x['exporter_commodity_reliability']),axis=1)

In [ ]:
trade_df = trade_df[['period','HS4_code','H0','exporterISO', 'importerISO','reconciled_value']]

save to csv file

In [ ]:
#trade_df.to_csv('reconciled7_trade_2022.csv', index=False)

In [ ]:
trade_df.head()

In [ ]:
# Group by exporter and sum the reconciled value
exporter_reconciled_value = trade_df.groupby('exporterISO')['reconciled_value'].sum()
exporter_reconciled_value.sort_values(ascending=False).reset_index().head(20)

In [ ]:
# Group by exporter and sum the reconciled value
importer_reconciled_value = trade_df.groupby('importerISO')['reconciled_value'].sum()
importer_reconciled_value.sort_values(ascending=False).head(10)

In [ ]:
total_value = trade_df['reconciled_value'].sum()
total_value

# Complexity Calculations

In [ ]:
trade_df_copy = trade_df.copy()

In [ ]:
# Applying thresholds
threshold_country = 1500000000
threshold_HS = 500000000
# Filter out countries
df_country =  trade_df_copy.groupby('exporterISO')['reconciled_value'].sum()
trade_df_copy = trade_df_copy[~trade_df_copy['exporterISO'].isin(df_country[df_country < threshold_country].index)]

# Filter out products (HS4)
df_HS = trade_df_copy.groupby('HS4_code')['reconciled_value'].sum()
trade_df_copy = trade_df_copy[~trade_df_copy['HS4_code'].isin(df_HS[df_HS < threshold_HS].index)]

In [ ]:
print(trade_df_copy['exporterISO'].nunique())
print(trade_df_copy['HS4_code'].nunique())

In [ ]:
comp = econci.Complexity(trade_df_copy, c='exporterISO', p='HS4_code', values='reconciled_value')
comp.calculate_indexes()

In [ ]:
eci = comp.eci
pci = comp.pci

In [ ]:
# create dataframes
eci = eci.rename(columns={0: "ECI"}).reset_index()
pci = pci.rename(columns={0: "PCI"}).reset_index()

In [ ]:
# display data
print(" Highest ECIs")
eci.sort_values(by='eci',ascending=False).head(20)

In [ ]:
# display data
print("5 Highest PCIs")
pci.sort_values(by='pci',ascending=False).head(10)